# 01 · Graph Construction — European Air Transport Network

> **Data vintage — June 2014.** This is a frozen OpenFlights route snapshot, *not* the current network. Every figure and number below describes 2014. (OpenFlights' route feed stopped updating in mid-2014; the topological questions this project asks — scale-free structure, community structure, percolation threshold — are structural invariants, so a dated snapshot is the standard and defensible choice.)

**This notebook builds the object every later notebook depends on.** It:

1. Loads the raw OpenFlights `airports.dat` and `routes.dat`.
2. Filters airports to a European bounding box and routes to **direct, operating** legs.
3. Builds a directed, weighted `DiGraph` and its undirected projection.
4. Reports the headline network statistics.
5. Renders the **hero route map** used as the README banner.

All loading/filtering/construction logic lives in `src/build_graph.py` (imported here, not redefined) so the pipeline is reusable and testable. Only the *presentation* map is notebook-specific.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook"   # emits a <div> + CDN plotly.js → renders in static HTML

%load_ext autoreload
%autoreload 2


def _find_src() -> Path:
    """Locate the repo's src/ dir so imports work regardless of launch cwd."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "build_graph.py").exists():
            return p / "src"
    raise FileNotFoundError("Could not locate src/ — run this from inside the repo.")


sys.path.insert(0, str(_find_src()))

import build_graph as bg
import utils

## 1 · Load and filter

Four modelling decisions are baked into `build_graph.py` — worth stating explicitly because they define what the graph *means*:

- **Europe bounding box** (lat 34–72°N, lon 25°W–45°E): continental Europe + UK + Cyprus, excluding Russia east of the Urals.
- **Nodes keyed by IATA code**, and *induced from route endpoints*. An airport that sits in the box but has no intra-European route never becomes a node — so the "isolated nodes" question is resolved by construction, not by a post-hoc drop.
- **Codeshare rows dropped** (`codeshare == 'Y'`): a codeshare is a flight *sold* by one carrier but *operated* by another. Keeping them would count one physical Lufthansa flight up to 5× via its Star Alliance resellers.
- **Non-stop only** (`stops == 0`): multi-leg rows would create phantom direct links.

**Edge weight = number of distinct *operating* carriers** on a (source → destination) pair. This is a service-level proxy — *not* traffic, frequency, or seats (routes.dat has no frequency data, so Barrat-style seat-weighting isn't possible; this limitation carries through the whole project).

In [ ]:
digraph, undirected, edges, airports_eu = bg.build_networks()

print(f"European airports in the bounding box (with IATA): {len(airports_eu)}")
print(f"... of which appear in >=1 direct EU route (nodes): {digraph.number_of_nodes()}")
print(f"Distinct directed edges: {digraph.number_of_edges()}")

edges.sort_values("weight", ascending=False).head()

## 2 · Network statistics

The bounding box holds ~1,000 airports, but only the subset with actual routes becomes the network — the rest are regional airfields with no scheduled European service in 2014. A handful of tiny components sit outside the giant component (isolated island/regional pairs); the largest connected component covers the overwhelming majority of nodes, which is exactly what we need for the centrality, community, and percolation analyses to come.

In [ ]:
summary = pd.DataFrame(
    {
        "DiGraph (directed)": bg.graph_summary(digraph),
        "Undirected projection": bg.graph_summary(undirected),
    }
).T
summary

In [ ]:
# Top hubs by degree (number of distinct destinations served).
# Note: this is degree, NOT passenger volume — which is why Stansted and the
# low-cost point-to-point bases outrank slot-constrained Heathrow here. That
# degree-vs-traffic distinction is the hook for the centrality analysis in NB02.
deg = dict(undirected.degree())
top = sorted(deg, key=deg.get, reverse=True)[:10]
pd.DataFrame(
    [
        {
            "IATA": c,
            "City": undirected.nodes[c]["city"],
            "Country": undirected.nodes[c]["country"],
            "Degree": deg[c],
        }
        for c in top
    ]
)

## 3 · Hero visualisation — the route map

A Plotly `Scattergeo` map: airports as markers (**size and colour ∝ degree**), routes as lines drawn in three **weight tiers** so that heavily-served links read as brighter/thicker than one-carrier routes. This is the README banner.

*(The map is presentation code, so it lives in the notebook rather than `src/`.)*

In [ ]:
# (min_weight, opacity, line_width) — faint single-carrier routes up to bold trunk links.
WEIGHT_TIERS = [(1, 0.05, 0.4), (2, 0.14, 0.7), (4, 0.35, 1.3)]


def _edge_segments(graph, wmin, wmax):
    """None-separated lon/lat arrays for edges with weight in [wmin, wmax)."""
    lon, lat = [], []
    for u, v, d in graph.edges(data=True):
        if wmin <= d["weight"] < wmax:
            lon += [graph.nodes[u]["lon"], graph.nodes[v]["lon"], None]
            lat += [graph.nodes[u]["lat"], graph.nodes[v]["lat"], None]
    return lon, lat


def route_map(graph):
    """Scattergeo route map: markers sized/coloured by degree, edges in weight tiers."""
    deg = dict(graph.degree())
    codes = list(graph.nodes())
    hover = [
        f"{c} — {graph.nodes[c]['city']}, {graph.nodes[c]['country']}<br>degree: {deg[c]}"
        for c in codes
    ]

    fig = go.Figure()

    bounds = [t[0] for t in WEIGHT_TIERS] + [np.inf]
    for i, (wmin, opacity, width) in enumerate(WEIGHT_TIERS):
        elon, elat = _edge_segments(graph, wmin, bounds[i + 1])
        fig.add_trace(go.Scattergeo(
            lon=elon, lat=elat, mode="lines",
            line=dict(width=width, color=utils.COLORS["route"]),
            opacity=opacity, hoverinfo="skip", showlegend=False,
        ))

    fig.add_trace(go.Scattergeo(
        lon=[graph.nodes[c]["lon"] for c in codes],
        lat=[graph.nodes[c]["lat"] for c in codes],
        mode="markers", text=hover, hoverinfo="text",
        marker=dict(
            size=[3 + 2.4 * np.sqrt(deg[c]) for c in codes],
            color=[deg[c] for c in codes], colorscale=utils.DEGREE_COLORSCALE,
            showscale=True, colorbar=dict(title="Degree", x=0.98, len=0.6),
            line=dict(width=0.3, color=utils.COLORS["bg"]), opacity=0.9,
        ),
        showlegend=False,
    ))

    fig.update_geos(
        scope="europe", resolution=50,
        showland=True, landcolor=utils.COLORS["land"],
        showocean=True, oceancolor=utils.COLORS["bg"],
        showcountries=True, countrycolor=utils.COLORS["coastline"],
        showcoastlines=True, coastlinecolor=utils.COLORS["coastline"],
        lataxis_range=[33, 71], lonaxis_range=[-25, 45],
        bgcolor=utils.COLORS["bg"],
    )
    fig.update_layout(
        title=dict(
            text=f"<b>Structure of European Aviation</b><br><sub>{utils.FIG_CAPTION} · "
                 f"{graph.number_of_nodes()} airports, {graph.number_of_edges()} links</sub>",
            x=0.5, xanchor="center", font=dict(color=utils.COLORS["text"], size=20),
        ),
        paper_bgcolor=utils.COLORS["bg"], plot_bgcolor=utils.COLORS["bg"],
        margin=dict(l=0, r=0, t=70, b=0), height=720,
    )
    return fig


fig = route_map(undirected)
fig.show()

In [ ]:
# Export the banner PNG for the README (scale=2 for a crisp raster).
# Static export needs kaleido:  conda install -c conda-forge python-kaleido
# Wrapped so restart-and-run-all never fails if kaleido/Chrome is absent.
utils.ensure_dir(utils.FIGURES_DIR)
out = utils.FIGURES_DIR / "route_map.png"
try:
    fig.write_image(out, scale=2)
    print("Saved", out)
except Exception as exc:  # noqa: BLE001 - export is optional at run time
    print(f"PNG export skipped ({type(exc).__name__}). Interactive map above is unaffected.")
    print("To enable: conda install -c conda-forge python-kaleido")

## Summary

**What we built.** A directed, weighted graph of the 2014 European air network — nodes = airports (IATA), edges = direct operating routes, weight = distinct operating carriers — plus its undirected projection for the community and small-world work. The giant component covers the vast majority of nodes, so the network is effectively connected for analysis.

**What stood out.** Ranking by *degree* surfaces low-cost point-to-point bases (Stansted, and the Iberian/Benelux hubs) above traffic giants like Heathrow — a first hint that **degree and traffic measure different things**, and that a hub's *structural* importance (does it connect otherwise-separate regions?) needs a different metric.

**Next — `02_centrality_stats.ipynb`.** Compute betweenness, closeness, eigenvector, and PageRank; test whether the degree distribution is scale-free (power-law fit); run the small-world test against an Erdős–Rényi null; and locate **Vienna's betweenness rank** — the Austrian angle, and where VIE's East–West bridge role should finally show up.